In [1]:
import numpy as np
from scipy import stats
from math import ceil
from pathlib import Path
import matplotlib.pyplot as plt
from contact_plots import get_data_from_files, get_cdf_of_data, do_chi_square_test
import warnings
warnings.filterwarnings('ignore')

In [2]:
data_folder = Path(r'C:\Users\martijnsparnaa\surfdrive\PhD\Activity scheduler\Simulations\2021_12\transfer\results_20211215')

contact_data_list, infra_and_measure_name_2_data = get_data_from_files(data_folder)

In [8]:
data_names_small = (
    '1.0_customer_customer_weight_over_contacts',
    '1.0_customer_customer_weight_over_agents',
    '1.0_customer_customer_contacts_over_agents',
    '1.0_staff_customer_weight_over_contacts',
    '1.0_staff_customer_weight_over_agents',
    '1.0_staff_customer_contacts_over_agents',
    None,
    '1.0_customer_staff_weight_over_agents',
    '1.0_customer_staff_contacts_over_agents',
)

data_names_middle = (
    '1.5_customer_customer_weight_over_contacts',
    '1.5_customer_customer_weight_over_agents',
    '1.5_customer_customer_contacts_over_agents',
    '1.5_staff_customer_weight_over_contacts',
    '1.5_staff_customer_weight_over_agents',
    '1.5_staff_customer_contacts_over_agents',
    None,
    '1.5_customer_staff_weight_over_agents',
    '1.5_customer_staff_contacts_over_agents',
)

data_names_large = (
    '2.0_customer_customer_weight_over_contacts',    
    '2.0_customer_customer_weight_over_agents',
    '2.0_customer_customer_contacts_over_agents',
    '2.0_staff_customer_weight_over_contacts',
    '2.0_staff_customer_weight_over_agents',
    '2.0_staff_customer_contacts_over_agents',
    None,
    '2.0_customer_staff_weight_over_agents',
    '2.0_customer_staff_contacts_over_agents',
)

measure_2_label = {
    'nn_nts_bvd':'Base',
    'n_nts_bvd':'Nbrh',
    'nn_ts_bvd':'Ts',
    'nn_nts_svd':'short',
    'nn_nts_lvd':'long'
}

base_measure_name = 'nn_nts_bvd'

In [19]:
%matplotlib wx

infra_name = 'base_case'

stacked_bar_data_name = '1.5_customer_staff_contacts_over_agents'
stacked_bar_cats = [1,2,3]

col_count = 3
row_count = int(ceil(len(data_names_small)/col_count))
fig, axes = plt.subplots(row_count, col_count)
axes = axes.flatten()
fig.suptitle(infra_name, fontsize=16)
fig.canvas.manager.set_window_title(infra_name)

lines = []
labels = []
for ax_nr, data_name in enumerate(data_names_middle):
    ax = axes[ax_nr]
    if data_name is None:
        continue    
    ax.grid()
    ax.set_title(data_name)
    box_data = []
    labels = []
    base_data = None
    bar_data = {category:[] for category in stacked_bar_cats}
    
    for measure_name, measure_label in measure_2_label.items():
        if data_name == stacked_bar_data_name:
            contact_data = infra_and_measure_name_2_data[infra_name][measure_name][data_name]
            contact_data_count = len(contact_data)
            for category in stacked_bar_cats:
                bar_data[category].append(np.sum(contact_data == category)/contact_data_count)
        else:            
            box_data.append(infra_and_measure_name_2_data[infra_name][measure_name][data_name])
            if measure_name == base_measure_name:
                base_data = box_data[-1]

        labels.append(measure_label)
        
    if data_name == stacked_bar_data_name:
        bottoms = None
                
        for category in stacked_bar_cats:
            if bottoms is None:
                ax.bar(labels, bar_data[category], 0.5, label=f'{category}')
                bottoms = bar_data[category]
            else:
                ax.bar(labels, bar_data[category], 0.5, label=f'{category}', bottom=bottoms)
                bottoms = [bottoms[ii] + bar_data[category][ii] for ii in range(len(labels))]
        ax.legend()
    else:
        ax.axhline(np.mean(base_data), linewidth=1, color='b', linestyle='-')
        ax.boxplot(box_data, vert=True, labels=labels, showmeans=True, meanline=True, meanprops={'linewidth':2, 'color':'r'})
    
plt.show()